## Questions commerciales

1. Quel produit cumule le plus grand nombre de ventes au total ?

2. Quel article est le plus/moins vendu parmi chacun des "departments" ?

3. Quel produit est généralement vendu avec chacun des articles les plus populaires ?

4. Quelle est la probabilité qu'un acheteur rachète un produit parmi une catégorie dont il a déjà commandé ?

5. Peut-on prédire la date de la prochaine commande d'un client en fonction de la fréquence à laquelle il commande ?

6. Est-ce que le jour et l'heure influence le nombre de vente parmi chaque "departement" ?

7. Certains articles sont-ils recommandés plus d'une fois ?

8. Existe-t-il une corrélation entre le premier et le deuxième article acheté ?

9. Pour quels jour et heure la catégorie alcool comporte le plus de vente ?

10. Peut-on prédire la taille d'un panier pour un client achetant de l'alcool ?

11. Parmi la catégorie "bakery", est-ce que les clients achètent plus de pain ("bread") le matin que de dessert ("bakery desserts") le midi et le soir ?

## Import

In [75]:
import pandas as pd
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import xml.sax.saxutils as saxutils
import html

## Lecture des CSV

In [76]:
df_order_products = pd.read_table("./datasets-project/datasets/order_products.csv", delimiter=",", low_memory = False)
df_orders = pd.read_table("./datasets-project/datasets/orders.csv", delimiter=",", low_memory = False)
df_products = pd.read_table("./datasets-project/datasets/products.csv", delimiter=",", low_memory= False)
df_aisles = pd.read_table("./datasets-project/datasets/aisles.csv", delimiter=",", low_memory= False)
df_departments = pd.read_table("./datasets-project/datasets/departments.csv", delimiter="," ,low_memory=False)

## Observations

In [77]:
# ---------------- order_product.csv ---------------- #

print(df_order_products.head(), "\n")
print(df_order_products.shape, "\n")
df_order_products.info(show_counts=True)


print(f"\nLes order_id sont compris entre 1 et {df_orders["order_id"].max()} dans orders.csv")
print("Nb de order_id inférieur à 1 et supérieur à 2999999 :", len(df_order_products[(df_order_products["order_id"] < 1) & (df_order_products["order_id"] > 2999999)]))
print("Nb de order_id n'étant pas des entiers :", len(df_order_products[df_order_products["order_id"].apply(lambda x: isinstance(x, int) == False)]), "\n")


print(f"Les product_id sont compris entre 1 et {df_products["product_id"].max()} dans products.csv")
print("Nb de product_id inférieur à 1 et supérieur à 49688 :", len(df_order_products[(df_order_products["product_id"] < 1) & (df_order_products["product_id"] > 49688)]))
print("Nb de product_id n'étant pas des entiers :", len(df_order_products[df_order_products["product_id"].apply(lambda x: isinstance(x, int) == False)]))
productId_filter_by_float = df_order_products[df_order_products["product_id"].apply(lambda x: "." in str(x))]
print("Nb de product_id avec une partie décimale :", len(productId_filter_by_float))
print("Nb de product_id avec une partie décimale non nulle :", len(productId_filter_by_float[productId_filter_by_float["product_id"].apply(lambda x: int(str(x).split(".")[1]) != 0)]), "\n")


print("Nb de add_to_cart_order inférieur à 1 :", len(df_order_products[df_order_products["add_to_cart_order"] < 1]))
print("Nb de add_to_cart_order n'étant pas des entiers :", len(df_order_products[df_order_products["add_to_cart_order"].apply(lambda x: isinstance(x, int) == False)]))
addToCartOrder_filter_by_float = df_order_products[df_order_products["add_to_cart_order"].apply(lambda x: "." in str(x))]
print("Nb de add_to_cart_order avec une partie décimale :", len(addToCartOrder_filter_by_float))
print("Nb de add_to_cart_order avec une partie décimale non nulle :", len(addToCartOrder_filter_by_float[addToCartOrder_filter_by_float["add_to_cart_order"].apply(lambda x: int(str(x).split(".")[1]) != 0)]))
article_ajouté_en_même_temps = df_order_products[
    df_order_products["order_id"].notna() & 
    df_order_products.duplicated(subset=["order_id", "add_to_cart_order"])
    ]
print("Nombre d'article ajouté en même temps dans le panier:", len(article_ajouté_en_même_temps), "\n")


print("Nb de reordered inférieur à 0 et supérieur à 1 :", len(df_order_products[(df_order_products["reordered"] < 0) & (df_order_products["reordered"] > 1)]))
print("Nb de reordered n'étant pas des entiers :", len(df_order_products[df_order_products["reordered"].apply(lambda x: isinstance(x, int) == False)]))
reordered_filter_by_float = df_order_products[df_order_products["reordered"].apply(lambda x: "." in str(x))]
print("Nb de reordered avec une partie décimale :", len(reordered_filter_by_float))
print("Nb de reordered avec une partie décimale non nulle :", len(reordered_filter_by_float[reordered_filter_by_float["reordered"].apply(lambda x: int(str(x).split(".")[1]) != 0)]))
df_order_products_userId = df_order_products.merge(
    df_orders[["order_id", "user_id"]],
    on="order_id",
    how="left"
)
df_order_products_userId["reordered_min"] = df_order_products_userId.groupby(["user_id", "product_id"])["reordered"].transform("min")
print("Nb de produit racheté sans achat initial :", len(df_order_products_userId[df_order_products_userId["reordered_min"] == 1]))

# ---------------------------------------------- #

   order_id  product_id  add_to_cart_order  reordered
0         2     33120.0                1.0        1.0
1         2     28985.0                2.0        1.0
2         2      9327.0                3.0        0.0
3         2     45918.0                4.0        1.0
4         2     30035.0                5.0        0.0 

(13692886, 4) 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13692886 entries, 0 to 13692885
Data columns (total 4 columns):
 #   Column             Non-Null Count     Dtype  
---  ------             --------------     -----  
 0   order_id           13692886 non-null  int64  
 1   product_id         13692437 non-null  float64
 2   add_to_cart_order  13692440 non-null  float64
 3   reordered          13692437 non-null  float64
dtypes: float64(3), int64(1)
memory usage: 417.9 MB

Les order_id sont compris entre 1 et 2999999 dans orders.csv
Nb de order_id inférieur à 1 et supérieur à 2999999 : 0
Nb de order_id n'étant pas des entiers : 0 

Les product_id sont com

In [78]:
# ---------------- orders.csv ---------------- #

print(df_orders.head(), "\n")
print(df_orders.shape, "\n")
df_orders.info()


print("\nNb de order_id inférieur à 1 :", len(df_orders[df_orders["order_id"] < 1]))
print("Nb de order_id n'étant pas des entiers :", len(df_orders[df_orders["order_id"].apply(lambda x: isinstance(x, int) == False)]))
if df_orders["order_id"].duplicated().any() == True:
    print("Doublons :", df_orders[df_orders["order_id"].duplicated()], "\n")
else:
    print("Il n'y a aucun order_id en double", "\n")


print("Nb de user_id inférieur à 1 :", len(df_orders[df_orders["user_id"] < 1]))
print("Nb de user_id n'étant pas des entiers :", len(df_orders[df_orders["user_id"].apply(lambda x: isinstance(x, int) == False)]))
#for k in range(len(df_orders.index)):
#    print(df_orders["user_id"].iloc[k])
userId_filter_by_float = df_orders[df_orders["user_id"].apply(lambda x: "." in str(x))]
print("Nb de user_id avec une partie décimale :", len(userId_filter_by_float))
print("Nb de user_id avec une partie décimale non nulle :", len(userId_filter_by_float[userId_filter_by_float["user_id"].apply(lambda x: int(str(x).split(".")[1]) != 0)]), "\n")


print("Nb de order_number inférieur à 1 :", len(df_orders[df_orders["order_number"] < 1]))
print("Nb de order_number n'étant pas des entiers :", len(df_orders[df_orders["order_number"].apply(lambda x: isinstance(x, int) == False)]))
orderNumber_filter_by_float = df_orders[df_orders["order_number"].apply(lambda x: "." in str(x))]
print("Nb de user_id avec une partie décimale :", len(orderNumber_filter_by_float))
print("Nb de user_id avec une partie décimale non nulle :", len(orderNumber_filter_by_float[orderNumber_filter_by_float["order_number"].apply(lambda x: int(str(x).split(".")[1]) != 0)]))
cmd_en_x2_par_user = df_orders[
    df_orders["user_id"].notna() & 
    df_orders.duplicated(subset=["user_id", "order_number"])
    ]
print("Nombre de commande en double par user_id :", len(cmd_en_x2_par_user), "\n")


print("Nb de order_dow inférieur à 1 et supérieur à 7 :", len(df_orders[(df_orders["order_dow"] < 1) & (df_orders["order_dow"] > 7)]))
print("Nb de order_dow n'étant pas des entiers :", len(df_orders[df_orders["order_dow"].apply(lambda x: isinstance(x, int) == False)]))
orderDow_filter_by_float = df_orders[df_orders["order_dow"].apply(lambda x: "." in str(x))]
print("Nb de order_dow avec une partie décimale :", len(orderDow_filter_by_float))
print("Nb de order_dow avec une partie décimale non nulle :", len(orderDow_filter_by_float[orderDow_filter_by_float["order_dow"].apply(lambda x: int(str(x).split(".")[1]) != 0)]), "\n")


print("Nb de order_hour_of_day inférieur à 0 et supérieur à 24 :", len(df_orders[(df_orders["order_hour_of_day"] < 0) & (df_orders["order_hour_of_day"] > 24)]))
print("Nb de order_hour_of_day valant 24 :", len(df_orders[df_orders["order_hour_of_day"] == 24]))
print("Nb de order_hour_of_day valant 0 :", len(df_orders[df_orders["order_hour_of_day"] == 0]))
print("Nb de order_hour_of_day n'étant pas des entiers :", len(df_orders[df_orders["order_hour_of_day"].apply(lambda x: isinstance(x, int) == False)]))
orderHourOfDay_filter_by_float = df_orders[df_orders["order_hour_of_day"].apply(lambda x: "." in str(x))]
print("Nb de order_hour_of_day avec une partie décimale :", len(orderHourOfDay_filter_by_float))
print("Nb de order_hour_of_day avec une partie décimale non nulle :", len(orderHourOfDay_filter_by_float[orderHourOfDay_filter_by_float["order_hour_of_day"].apply(lambda x: int(str(x).split(".")[1]) != 0)]), "\n")


print("Nb de days_since_prior_order inférieur à 0 :", len(df_orders[df_orders["days_since_prior_order"] < 0]))
print("Nb de days_since_prior_order n'étant pas des entiers :", len(df_orders[df_orders["days_since_prior_order"].apply(lambda x: isinstance(x, int) == False)]))
daysSincePriorOrder_filter_by_float = df_orders[df_orders["days_since_prior_order"].apply(lambda x: "." in str(x))]
print("Nb de days_since_prior_order avec une partie décimale :", len(daysSincePriorOrder_filter_by_float))
print("Nb de days_since_prior_order avec une partie décimale non nulle :", len(daysSincePriorOrder_filter_by_float[daysSincePriorOrder_filter_by_float["days_since_prior_order"].apply(lambda x: int(str(x).split(".")[1]) != 0)]), "\n")

# ---------------------------------------------- #

   order_id  user_id  order_number  order_dow  order_hour_of_day  \
0   2539329      1.0           1.0        2.0                8.0   
1   2398795      1.0           2.0        3.0                7.0   
2    473747      1.0           3.0        3.0               12.0   
3   2254736      1.0           4.0        4.0                7.0   
4    431534      1.0           5.0        4.0               15.0   

   days_since_prior_order  
0                     NaN  
1                    15.0  
2                    21.0  
3                    29.0  
4                    28.0   

(1444444, 6) 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1444444 entries, 0 to 1444443
Data columns (total 6 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   order_id                1444444 non-null  int64  
 1   user_id                 1444413 non-null  float64
 2   order_number            1444421 non-null  float64
 3   order_dow     

In [79]:
# ---------------- products.csv ---------------- #

print(df_products.head(), "\n")
print(df_products.shape, "\n")
df_products.info()


print("\nNb de product_id inférieur à 1 :", len(df_products[df_products["product_id"] < 1]))
print("Nb de product_id n'étant pas des entiers :", len(df_products[df_products["product_id"].apply(lambda x: isinstance(x, int) == False)]))
if df_products["product_id"].duplicated().any() == True:
    print("Doublons :", df_products[df_products["product_id"].duplicated()], "\n")
else:
    print("Il n'y a aucun product_id en double", "\n")


print("Nb de product_name contenant un '+' :", len(df_products[df_products["product_name"].apply(lambda x: "+" in str(x))]))
df_products_plus = df_products[df_products["product_name"].apply(lambda x: "+" in str(x))]
for k in range(10):
    print(df_products_plus['product_name'].iloc[k])
print("\nNb de product_name contenant un '&' ou 'and' :", len(df_products[df_products["product_name"].apply(lambda x: "&" in str(x) or "and" in str(x))]))
df_products_and = df_products[df_products["product_name"].apply(lambda x: "&" in str(x) or "and" in str(x))]
for k in range(10):
    print(df_products_and['product_name'].iloc[k])
print("\nNb de product_name contenant des guillemets :", len(df_products[df_products["product_name"].apply(lambda x: str(x).strip().startswith("'") and str(x).strip().endswith("'"))]))
print("Nb de product_name commencant par un espace :", len(df_products[df_products["product_name"].apply(lambda x: str(x).startswith(" "))]))
'''
Débuggage 
for k in range(len(df_products.index)):
    if k == 74:
        print(df_products["product_id"].iloc[k], str(df_products["product_name"].iloc[k]).strip()) #product_name étant normalement entouré de guillemets
'''
print("\nNb de product_name contenant des caractères spéciaux :", len(df_products[df_products["product_name"].apply(lambda x: re.compile(r'[@_!#$%^*()<>?/\|}{~:"]').search(str(x)) != None)]))
df_products_specialChart = df_products[df_products["product_name"].apply(lambda x: re.compile(r'[@_!#$%^*()<>?/\|}{~:"]').search(str(x)) != None)]
for k in range(10):
    print(df_products_specialChart['product_name'].iloc[k])
print("\nHTML à décoder :")
for k in range(len(df_products["product_name"])):
    if str(html.unescape(df_products["product_name"].iloc[k])) != str(df_products["product_name"].iloc[k]):
        print(df_products["product_name"].iloc[k])
        print(html.unescape(df_products["product_name"].iloc[k]), "\n")
if df_products[df_products["product_name"].apply(lambda x: str(x).lower()).duplicated()]["product_name"].any() == True:
    print("\nDoublons :", df_products[df_products["product_name"].apply(lambda x: str(x).lower()).duplicated()], "\n")
else:
    print("\nIl n'y a aucun product_name en double", "\n")


print("Nb de aisle_id inférieur à 1 :", len(df_products[df_products["aisle_id"] < 1]))
print("Nb de aisle_id supérieur à 134 :", len(df_products[df_products["aisle_id"] > 134]))
print("Nb de aisle_id n'étant pas des entiers :", len(df_products[df_products["aisle_id"].apply(lambda x: isinstance(x, int) == False)]), "\n")


print("Nb de department_id inférieur à 1 :", len(df_products[df_products["department_id"] < 1]))
print("Nb de department_id supérieur à 21 :", len(df_products[df_products["department_id"] > 21]))
print("Nb de department_id n'étant pas des entiers :", len(df_products[df_products["department_id"].apply(lambda x: isinstance(x, int) == False)]))

# ---------------------------------------------- #

   product_id                                       product_name  aisle_id  \
0           1                         Chocolate Sandwich Cookies        61   
1           2                                   All-Seasons Salt       104   
2           3               Robust Golden Unsweetened Oolong Tea        94   
3           4  Smart Ones Classic Favorites Mini Rigatoni Wit...        38   
4           5                          Green Chile Anytime Sauce         5   

   department_id  
0             19  
1             13  
2              7  
3              1  
4             13   

(49688, 4) 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49688 entries, 0 to 49687
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   product_id     49688 non-null  int64 
 1   product_name   49688 non-null  object
 2   aisle_id       49688 non-null  int64 
 3   department_id  49688 non-null  int64 
dtypes: int64(3), object(1)
memory u

In [80]:
# ---------------- Questions commerciales ---------------- #

#Question 11 - Trouver les sous-catégories de 'bakery'
df_products_bakery = df_products[df_products["department_id"] == 3]
aisles_id = list(df_products_bakery["aisle_id"].unique())
aisle_names = []
for aisle_id in aisles_id:
    for k in range(len(df_aisles.index)):
        if df_aisles["aisle_id"].iloc[k] == int(aisle_id):
            if df_aisles["aisle"].iloc[k] not in aisle_names:
                aisle_names.append(df_aisles["aisle"].iloc[k])
print("Les sous catégories de 'bakery' sont :", ", ".join(aisle_names))

# ---------------------------------------------- #

Les sous catégories de 'bakery' sont : tortillas flat bread, bread, breakfast bakery, bakery desserts, buns rolls


#### Interprétation des datasets

- "product.csv" permet de créer les fiches produits en associant à chaque élément une catégorie, "department", puis une sous catégorie, "aisle" : DEPARTEMENT ---> AISLE ---> PRODUCT

- "order.csv", donne l'id du client ainsi que les informations sur la commande sans donner ce que comporte le panier

- "oder_product.csv" ---> fait la liaison entre les commandes effectuées et leur panier, "order_id" fournit les informations de la commande et "product_id" y associe un produit


## Nettoyage

In [81]:
# ---------------- Clean order products ---------------- #

df_order_products_clean = df_order_products.dropna()
df_order_products_clean = df_order_products_clean.astype({"order_id": int, "product_id": int, "add_to_cart_order": int, "reordered": int})

# ------------------------------------------------------ #

# ---------------- Clean orders ---------------- #

df_orders_clean = df_orders.dropna()
df_orders_clean = df_orders_clean.astype({"order_id": int, "user_id": int, "order_number": int, "order_dow": int, "order_hour_of_day": int, "days_since_prior_order": int})

# ---------------------------------------------- #

# ---------------- Clean products ---------------- #

df_products_clean = df_products.astype({"product_id": int, "product_name": str, "aisle_id": int, "department_id": int})
df_products_clean["product_name"] = df_products_clean["product_name"].apply(lambda x: html.unescape(x))
df_products_clean["lower_product_name"] = df_products_clean["product_name"].apply(lambda x: x.lower())
df_products_clean = df_products_clean.drop_duplicates(subset=["product_id", "lower_product_name", "aisle_id", "department_id"])
df_products_clean = df_products_clean.drop(columns="lower_product_name")

# ------------------------------------------------ #

# ---------------- Fusion ---------------- #

# Étape 1, merge des aisles et department ID:

df_products_clean = df_products_clean.merge(df_aisles, how = "inner", on="aisle_id")
df_products_clean = df_products_clean.merge(df_departments, how = "inner", on="department_id")
df_products_clean = df_products_clean.drop(columns=["aisle_id", "department_id"])
#print(df_products_clean.head())

# ------ #

# Étape 2, merge des order et product ID:

df_order_products_clean = df_order_products_clean.merge(df_orders_clean, how = "inner", on="order_id")
df_order_products_clean = df_order_products_clean.merge(df_products_clean, how = "inner", on="product_id")
df_order_products_clean = df_order_products_clean.drop(columns=["order_id", "product_id"])
print(df_order_products_clean.head())
print(df_order_products_clean.shape)

# ---------------------------------------- #

   add_to_cart_order  reordered  user_id  order_number  order_dow  \
0                  1          1   202279             3          5   
1                  2          1   202279             3          5   
2                  3          0   202279             3          5   
3                  4          1   202279             3          5   
4                  5          0   202279             3          5   

   order_hour_of_day  days_since_prior_order           product_name  \
0                  9                       8     Organic Egg Whites   
1                  9                       8  Michigan Organic Kale   
2                  9                       8          Garlic Powder   
3                  9                       8         Coconut Butter   
4                  9                       8      Natural Sweetener   

                aisle  department  
0                eggs  dairy eggs  
1    fresh vegetables     produce  
2   spices seasonings      pantry  
3       oils v

## Export des CSV nettoyés

In [82]:
if not os.path.exists("./datasets-cleaned"):
    os.makedirs("./datasets-cleaned")

df_order_products_clean.to_csv("./datasets-cleaned/df_order_products_clean.csv", index=False)